# Qwen3.5-9B — Interactive Chat on Colab Free Tier (T4 GPU)

This notebook runs **Qwen3.5-9B**, quantized to GGUF (4-bit, `UD-Q4_K_XL`), fully offloaded to the GPU using `llama-cpp-python`. The quantized weights are ~6 GB, which comfortably fits on a free-tier T4 (15 GB VRAM) alongside a reasonable context window.

**A couple of things worth knowing before you start:**
- **Runtime**: Go to `Runtime → Change runtime type → T4 GPU` before running anything below.
- **Thinking model**: Qwen3.5 reasons inside `<think>...</think>` tags before giving its final answer by default. This notebook asks the model to skip that (non-thinking / instruct-style replies) and also strips any leftover `<think>` tags from what's displayed, so you get direct answers.
- **First run is slow**: compiling `llama-cpp-python` with CUDA support takes a few minutes, and downloading the ~6 GB model takes a few more. After that, cells re-run fast.
- Model card: [unsloth/Qwen3.5-9B-GGUF](https://huggingface.co/unsloth/Qwen3.5-9B-GGUF)


## 1. Confirm you have a GPU

If this errors or shows no GPU, go to `Runtime → Change runtime type` and select **T4 GPU**, then re-run.

In [1]:
!nvidia-smi

Wed Aug 19 12:39:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install `llama-cpp-python` with CUDA support

Built from source against Colab's CUDA toolkit so the model actually runs on the GPU (the plain `pip install llama-cpp-python` gives you a CPU-only build, which will be very slow for a 9B model). This step takes roughly 5–10 minutes — it only needs to run once per Colab session.

In [2]:
import os

os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"

!apt-get -qq update
!apt-get -qq install -y cmake ninja-build

!pip install -U pip

!pip install \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
    llama-cpp-python \
    huggingface_hub

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 122492 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.10.1-1_amd64.deb ...
Unpacking ninja-build (1.10.1-1) ...
Setting up ninja-build (1.10.1-1) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 GB 19.2 MB/s  0:00:33
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [6]:
from llama_cpp import llama_cpp
print("CUDA-enabled build:", llama_cpp.llama_supports_gpu_offload())

CUDA-enabled build: True


In [7]:
# Sanity check: confirm the build actually picked up CUDA support
from llama_cpp import llama_cpp
print("CUDA-enabled build:", llama_cpp.llama_supports_gpu_offload())

CUDA-enabled build: True


## 3. Download the quantized model

We use Unsloth's **Dynamic 2.0** GGUF quantization of Qwen3.5-9B at `UD-Q4_K_XL` (~6 GB) — a 4-bit quant that keeps key layers at higher precision for better quality than a naive 4-bit quant, while still being small enough for the free-tier T4 and Colab's disk/RAM limits.

If you hit disk or RAM issues, drop to a smaller quant such as `UD-Q3_K_XL` (~5 GB) or `UD-Q2_K_XL` (~4.1 GB) by changing `QUANT` below — quality drops a bit as you go smaller, but it'll run on tighter setups.

In [8]:
from huggingface_hub import hf_hub_download

REPO_ID = "unsloth/Qwen3.5-9B-GGUF"
QUANT = "UD-Q4_K_XL"   # ~5.97 GB. Alternatives: UD-Q3_K_XL (~5.05 GB), UD-Q2_K_XL (~4.12 GB)

from huggingface_hub import list_repo_files
matches = [f for f in list_repo_files(REPO_ID) if QUANT in f and f.endswith(".gguf")]
assert matches, f"No GGUF file found for quant '{QUANT}' in {REPO_ID}"
filename = matches[0]
print("Downloading:", filename)

model_path = hf_hub_download(repo_id=REPO_ID, filename=filename)
print("Saved to:", model_path)

Downloading: Qwen3.5-9B-UD-Q4_K_XL.gguf
Saved to: /root/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B-GGUF/snapshots/3885219b6810b007914f3a7950a8d1b469d598a5/Qwen3.5-9B-UD-Q4_K_XL.gguf


## 4. Load the model

- `n_gpu_layers=-1` offloads every layer to the GPU.
- `n_ctx=8192` is a comfortable context length for the free-tier T4. Qwen3.5 natively supports up to 262,144 tokens, but a larger `n_ctx` means a larger KV cache in VRAM — raise this only if you have headroom left (check with `!nvidia-smi` after loading) and lower it if you hit an out-of-memory error.

In [9]:
from llama_cpp import Llama

llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # offload all layers to the T4
    n_ctx=8192,           # context window; raise/lower based on available VRAM
    n_batch=512,
    flash_attn=True,
    verbose=False,
)
print("Model loaded.")

Model loaded.


In [10]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

memory.used [MiB], memory.total [MiB]
6049 MiB, 15360 MiB


## 5. Chat helper

This wraps `create_chat_completion`, streams tokens as they're generated, and strips `<think>...</think>` reasoning blocks from what gets displayed so you see clean, direct answers. Sampling parameters follow Qwen's recommended **instruct (non-thinking) mode** settings.

In [13]:
import re

SYSTEM_PROMPT = """
You are an AI children's story writer.

Your task is to create meaningful, age-appropriate stories
based on a child's everyday experience.

The story should:
- Reflect the emotional situation in the provided event.
- Help the child explore the selected story goal through the story.
- Use age-appropriate vocabulary, sentence length, and complexity.
- Have a clear beginning, middle, and ending.
- Show emotions through the character's experiences and actions.
- Provide a gentle and positive resolution.
- Be enjoyable and engaging when read aloud by a parent.
- Avoid frightening, violent, traumatic, or inappropriate content.
- Avoid directly giving psychological or medical advice.
- Avoid forcing an obvious moral or lesson.
- Treat the child's event as inspiration rather than copying private
  details unnecessarily.

The story should feel natural and meaningful, not like an educational
lesson disguised as a story.
"""

def chat_stream(llm, messages, max_tokens=2048):
    """Streams a reply, hides <think>...</think> content, prints/returns the rest."""
    stream = llm.create_chat_completion(
    messages=messages,
    max_tokens=max_tokens,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    repeat_penalty=1.0,
    presence_penalty=1.5,
    stream=True,
)

    full_text = ""
    in_think = False
    visible_buffer = ""

    for chunk in stream:
        delta = chunk["choices"][0]["delta"].get("content", "")
        if not delta:
            continue
        full_text += delta

        # Track whether we're currently inside a <think> block so we don't print it.
        buf = full_text
        # Strip any complete <think>...</think> blocks and drop an unfinished trailing one.
        visible = re.sub(r"<think>.*?</think>", "", buf, flags=re.DOTALL)
        visible = re.sub(r"<think>.*$", "", visible, flags=re.DOTALL)

        new_text = visible[len(visible_buffer):]
        if new_text:
            print(new_text, end="", flush=True)
            visible_buffer = visible

    print()  # trailing newline
    # Final cleaned response (used for conversation history)
    final_reply = re.sub(r"<think>.*?</think>", "", full_text, flags=re.DOTALL).strip()
    return final_reply

## 6. Interactive chat loop

Run this cell and type your messages at the prompt. Type `exit`, `quit`, or leave the input blank to stop. Conversation history is kept for the duration of the loop so the model has context from earlier turns.

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Qwen3.5-9B interactive chat — type 'exit' to quit.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        break

    if not user_input or user_input.lower() in {"exit", "quit"}:
        print("Ending chat.")
        break

    messages.append({"role": "user", "content": user_input})

    print("Qwen: ", end="")
    reply = chat_stream(llm, messages)
    messages.append({"role": "assistant", "content": reply})
    print()

Qwen3.5-9B interactive chat — type 'exit' to quit.

You: Create a bedtime story using these details:  Child age: 7 Daily event: Sara was nervous on her first day at school. Story goal: Confidence Main character: A little rabbit Language: English  The story should help the child explore the feeling of being nervous about something new and gradually develop confidence.
Qwen: **The Little Rabbit Who Learned to Hop Forward**

Once upon a time, in the middle of a very green and very tall forest, lived a little rabbit named Sara.

Sara had soft gray fur that felt like clouds against her nose, and ears that could swivel to hear even the tiniest rustle of leaves. She loved jumping through dandelions, chasing fireflies at dusk, and napping in the sunniest patch of clover. But today was different. Today was a big new thing: it was Sara's very first day at Forest School.

As soon as her mommy rabbit tucked her into her tiny backpack, Sara's tummy gave a little flip-flop. *Thump-thump, thump-thump

## Notes & troubleshooting

- **Out of memory when loading**: lower `n_ctx` in step 4 (e.g. `4096`), or switch to a smaller quant in step 3 (`UD-Q3_K_XL` or `UD-Q2_K_XL`).
- **Slow generation**: confirm step 2's sanity check printed `True` for `llama_supports_gpu_offload()` — if it printed `False`, the build didn't pick up CUDA and you're running on CPU. Re-run step 2 after restarting the runtime.
- **Session disconnects**: Colab free tier has usage limits and will disconnect idle or long-running sessions; the model download and compiled wheel aren't preserved between sessions, so re-running from the top is normal.
- **Resetting the conversation**: re-run the cell in step 6 (it resets `messages` back to just the system prompt).
- **Want the model's reasoning visible?** Remove the `chat_template_kwargs={"enable_thinking": False}` line in step 5 and skip the `<think>` stripping if you'd rather see Qwen3.5's chain-of-thought.
